# Решение: детектор ботов по кукам

Обучение, валидация и сабмит. Признаки — `features.py`, метрика — `metric.py`.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys
PROJECT_DIR = "/content/drive/MyDrive/avito"
os.chdir(PROJECT_DIR)

sys.path.insert(0, PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, "notebooks"))

!pip install -q catboost

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, average_precision_score
from catboost import CatBoostClassifier, Pool

from metric import pr_curve, precision_at_recall, recall_at_fpr
from features import build_features


## 1. Загрузка и предобработка

In [3]:
#DATA_PATH = "/kaggle/input/datasets/sluchevskiipersii/cookies/data/"
DATA_PATH = "/content/drive/MyDrive/avito/data"
PLATFORM_MAP = {"web": "web", "desktop": "web", "android": "android", "ios": "ios", "iphone": "ios"}
TS = ["cookie_created_at", "window_start_ts", "window_end_ts"]
SEED = 42

train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"), parse_dates=TS)
test = pd.read_csv(os.path.join(DATA_PATH, "test.csv"), parse_dates=TS)
events = pd.read_csv(os.path.join(DATA_PATH, "events.csv.gz"), parse_dates=["event_ts"])

print(f"train {train.shape} | test {test.shape} | events {events.shape}")


train (11091, 5) | test (4909, 4) | events (328905, 14)


In [4]:
# platform и удаление полных дублей
events["platform"] = events["platform"].str.lower().map(PLATFORM_MAP)
events = events.drop_duplicates().reset_index(drop=True)

# оставляем только события внутри окна наблюдения каждой куки
meta = pd.concat([train.assign(split="train"), test.assign(split="test")], ignore_index=True)
ev = events.merge(
    meta[["cookie_id", "window_start_ts", "window_end_ts", "split", "target"]], on="cookie_id", how="inner"
)
in_window = (ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)
ev = ev[in_window].reset_index(drop=True)
print("Событий в окне:", len(ev))


Событий в окне: 283830


## 2. Признаки

In [5]:
feats_v2 = build_features(meta, ev)
tr2 = feats_v2[feats_v2.split == "train"].copy()
te2 = feats_v2[feats_v2.split == "test"].copy()

print("train:", tr2.shape, "test:", te2.shape)
print("Доля ботов в train:", tr2.target.mean().round(4))
print("Куки без событий: train", (tr2.n_events == 0).sum(), "test", (te2.n_events == 0).sum())

train: (11091, 100) test: (4909, 100)
Доля ботов в train: 0.0811
Куки без событий: train 0 test 0


In [6]:
DROP = ["split", "target", "window_start_ts", "window_end_ts", "cookie_created_at"]

CAT = ["ua_browser", "ua_os", "platform"]
NUM = [c for c in tr2.select_dtypes("number").columns
       if c not in DROP and c not in CAT]
FEATURES = NUM + CAT

X = tr2[FEATURES].copy()
y = tr2["target"].astype(int).values
X_test = te2[FEATURES].copy()

print(f"Всего фичей: {len(FEATURES)} (numeric={len(NUM)}, cat={len(CAT)})")

Всего фичей: 93 (numeric=90, cat=3)


## 3. Валидация по времени и обучение CatBoost

In [7]:
tr2_sorted = tr2.sort_values("window_start_ts").copy()

val_days = 3  # хвостовые даты уходят в валидацию
cut_date = tr2_sorted["window_start_ts"].dt.date.max() - pd.Timedelta(days=val_days - 1)
cut_ts = tr2_sorted["window_start_ts"].max().normalize() - pd.Timedelta(days=val_days - 1)

is_val = tr2_sorted["window_start_ts"] >= cut_ts
tr_fold = tr2_sorted.loc[~is_val].copy()
va_fold = tr2_sorted.loc[is_val].copy()

print(f"cut_ts={cut_ts.date()}")
print(f"train fold: {len(tr_fold)} (bots={tr_fold.target.sum()}), "
      f"val fold: {len(va_fold)} (bots={va_fold.target.sum()})")
print("train bots share:", tr_fold.target.mean().round(4),
      "| val bots share:", va_fold.target.mean().round(4))

cut_ts=2026-04-17
train fold: 9140 (bots=739.0), val fold: 1951 (bots=160.0)
train bots share: 0.0809 | val bots share: 0.082


In [8]:
from catboost import CatBoostClassifier, Pool

X_tr = tr_fold[FEATURES].copy()
y_tr = tr_fold["target"].astype(int).values
X_va = va_fold[FEATURES].copy()
y_va = va_fold["target"].astype(int).values


train_pool = Pool(X_tr, y_tr, cat_features=CAT)
val_pool = Pool(X_va, y_va, cat_features=CAT)

model = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=6.0,
    random_seed=SEED,
    eval_metric="PRAUC",
    od_type="Iter",
    od_wait=200,
    loss_function="Logloss",
    verbose=200,
    class_weights=None,
)
model.fit(train_pool, eval_set=val_pool, use_best_model=True)

val_score = model.predict_proba(X_va)[:, 1]

# Метрики
p_at_r = precision_at_recall(y_va, val_score, recall=0.70)
r_at_fpr = recall_at_fpr(y_va, val_score, fpr=0.01)
roc = roc_auc_score(y_va, val_score)
pr = average_precision_score(y_va, val_score)

print(f"\nP@R>=0.70 : {p_at_r:.4f}")
print(f"Recall@FPR<=1%: {r_at_fpr:.4f}")
print(f"ROC-AUC   : {roc:.4f}")
print(f"PR-AUC    : {pr:.4f}")

0:	learn: 0.4449774	test: 0.3594722	best: 0.3594722 (0)	total: 84.8ms	remaining: 4m 14s
200:	learn: 0.8486929	test: 0.7543638	best: 0.7543638 (200)	total: 7.46s	remaining: 1m 43s
400:	learn: 0.9065854	test: 0.7677825	best: 0.7690161 (352)	total: 16s	remaining: 1m 43s
600:	learn: 0.9382610	test: 0.7726258	best: 0.7726812 (580)	total: 23.9s	remaining: 1m 35s
800:	learn: 0.9611897	test: 0.7775339	best: 0.7775339 (800)	total: 31.8s	remaining: 1m 27s
1000:	learn: 0.9758402	test: 0.7819515	best: 0.7825077 (985)	total: 40.3s	remaining: 1m 20s
1200:	learn: 0.9856170	test: 0.7835860	best: 0.7844632 (1174)	total: 47.8s	remaining: 1m 11s
1400:	learn: 0.9917486	test: 0.7840736	best: 0.7845496 (1313)	total: 56.8s	remaining: 1m 4s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.7845496412
bestIteration = 1313

Shrink model to first 1314 iterations.

P@R>=0.70 : 0.7244
Recall@FPR<=1%: 0.6000
ROC-AUC   : 0.9288
PR-AUC    : 0.7850


In [9]:
imp = pd.DataFrame({
    "feature": FEATURES,
    "importance": model.get_feature_importance(train_pool),
}).sort_values("importance", ascending=False)

display(imp.head(25).round(3))

# сколько фичей дают 95% важности
cum = imp.importance.cumsum() / imp.importance.sum()
n95 = int((cum < 0.95).sum() + 1)
print(f"\n{n95} фичей дают 95% важности")

,feature,importance
3,gap_median,6.885
67,ptr_x_std,6.075
41,item_uniq_ratio,5.302
43,cat_uniq_ratio,4.344
6,gap_min,3.853
45,loc_uniq_ratio,3.425
53,mean_page,2.649
69,step_mean,2.609
42,cat_top_share,2.588
8,gap_p90,2.525



66 фичей дают 95% важности


## 4. Кросс-валидация по времени

In [10]:
unique_dates = np.sort(tr2["window_start_ts"].dt.normalize().unique())
print(f"Уникальных дат в train: {len(unique_dates)}: {[str(pd.Timestamp(d).date()) for d in unique_dates]}")

Уникальных дат в train: 14: ['2026-04-06', '2026-04-07', '2026-04-08', '2026-04-09', '2026-04-10', '2026-04-11', '2026-04-12', '2026-04-13', '2026-04-14', '2026-04-15', '2026-04-16', '2026-04-17', '2026-04-18', '2026-04-19']


In [11]:
# TimeSeriesSplit работает по индексам, поэтому дадим ему группы по датам
dates_sorted = tr2.sort_values("window_start_ts").reset_index(drop=True)
date_codes = dates_sorted["window_start_ts"].dt.normalize().astype("int64").values

n_splits = 3
tscv = TimeSeriesSplit(n_splits=n_splits)

results = []
for fold, (idx_tr, idx_va) in enumerate(tscv.split(dates_sorted)):
    tr_f = dates_sorted.iloc[idx_tr].copy()
    va_f = dates_sorted.iloc[idx_va].copy()
    # не хотим, чтобы дни пересекались — обрежем
    max_tr_date = tr_f["window_start_ts"].max()
    va_f = va_f[va_f["window_start_ts"] > max_tr_date]
    if len(va_f) == 0 or va_f["target"].nunique() < 2:
        continue

    Xtr = tr_f[FEATURES].copy()
    Xva = va_f[FEATURES].copy()
    ytr = tr_f["target"].astype(int).values
    yva = va_f["target"].astype(int).values
    for c in CAT:
        Xtr[c] = Xtr[c].fillna("__nan__").astype(str)
        Xva[c] = Xva[c].fillna("__nan__").astype(str)

    m = CatBoostClassifier(
        iterations=1500, learning_rate=0.05, depth=6, l2_leaf_reg=6.0,
        random_seed=SEED, eval_metric="PRAUC",
        od_type="Iter", od_wait=150, verbose=0,
    )
    m.fit(Xtr, ytr, cat_features=CAT)
    p = m.predict_proba(Xva)[:, 1]

    results.append({
        "fold": fold,
        "n_val": len(va_f),
        "bots_val": int(yva.sum()),
        "P@R>=0.70": precision_at_recall(yva, p, 0.70),
        "Recall@FPR<=1%": recall_at_fpr(yva, p, 0.01),
        "ROC-AUC": roc_auc_score(yva, p),
        "PR-AUC": average_precision_score(yva, p),
    })

cv_res = pd.DataFrame(results)
display(cv_res.round(4))
print("\nСреднее по фолдам:")
display(cv_res.drop(columns=["fold"]).mean().round(4).to_frame().T)

,fold,n_val,bots_val,P@R>=0.70,Recall@FPR<=1%,ROC-AUC,PR-AUC
0,0,2242,200,0.7157,0.5800,0.9162,0.7484
1,1,2389,180,0.7079,0.5944,0.9209,0.7609
2,2,2691,220,0.6754,0.5636,0.9210,0.7568



Среднее по фолдам:


,n_val,bots_val,P@R>=0.70,Recall@FPR<=1%,ROC-AUC,PR-AUC
0,2440.6667,200.0,0.6997,0.5794,0.9193,0.7554


## 5. Финальная модель и сабмит

In [ ]:
X_full = tr2[FEATURES].copy()
y_full = tr2["target"].astype(int).values
X_test_final = X_test.copy()
for c in CAT:
    X_full[c] = X_full[c].fillna("__nan__").astype(str)
    X_test_final[c] = X_test_final[c].fillna("__nan__").astype(str)

final_model = CatBoostClassifier(
    iterations=getattr(model, "best_iteration_", None) or 1200,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=6.0,
    random_seed=SEED,
    eval_metric="PRAUC",
    verbose=200,
)
final_model.fit(X_full, y_full, cat_features=CAT)

test_score = final_model.predict_proba(X_test_final)[:, 1]
print("test score:", pd.Series(test_score).describe().round(4).to_dict())

0:	learn: 0.3388576	total: 36.9ms	remaining: 48.4s
200:	learn: 0.8427188	total: 10.4s	remaining: 57.6s
400:	learn: 0.8927641	total: 20.1s	remaining: 45.8s
600:	learn: 0.9269029	total: 28.5s	remaining: 33.7s
800:	learn: 0.9487077	total: 38.6s	remaining: 24.7s
1000:	learn: 0.9632400	total: 46.6s	remaining: 14.5s


In [ ]:
submission = pd.DataFrame({
    "cookie_id": te2.index,
    "score": test_score,
})

submission = submission.drop_duplicates("cookie_id").set_index("cookie_id").reindex(test.cookie_id).reset_index()
submission["score"] = submission["score"].clip(0, 1).fillna(0.0)


submission.to_csv("submission.csv", index=False)
print(submission.head())
print("submission.csv", submission.shape)

## 6. PR-кривая на отложенной валидации

In [ ]:
import matplotlib.pyplot as plt

prec, rec = pr_curve(y_va, val_score)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(rec, prec, lw=2, color="#4c72b0")
ax.axvline(0.70, ls="--", color="k", lw=1)
ax.axhline(p_at_r, ls=":", color="#dd8452", lw=1,
           label=f"P@R>=0.70 = {p_at_r:.3f}")
ax.set(xlabel="Recall", ylabel="Precision",
       title="PR-кривая на отложенной валидации")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()